# visionProyecto — Entrenar CNN de regresión (FreiHAND) en GPU

CNN desde cero que predice las **5 flexiones por dedo** (regresión), entrenada con FreiHAND.

**Antes de correr:** menú `Entorno de ejecución` → `Cambiar tipo de entorno` → **GPU (T4)**.
Luego `Entorno de ejecución` → **Ejecutar todo**.

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
!nvidia-smi -L

In [ ]:
%cd /content
![ -d visionProyecto ] || git clone https://github.com/dake14/visionProyecto.git
%cd /content/visionProyecto

In [ ]:
# Descarga FreiHAND (~3.6 GB) con la red de Google y lo descomprime
import glob, os, shutil

ZIP = '/content/FreiHAND_pub_v2.zip'
DEST = '/content/visionProyecto/datasets/FreiHAND_pub_v2'

if not os.path.exists(os.path.join(DEST, 'training_xyz.json')):
    if not os.path.exists(ZIP):
        !wget -q --show-progress https://lmb.informatik.uni-freiburg.de/data/freihand/FreiHAND_pub_v2.zip -O {ZIP}
    !unzip -q -o {ZIP} -d /content/visionProyecto/datasets/_tmp_freihand
    xyz = glob.glob('/content/visionProyecto/datasets/_tmp_freihand/**/training_xyz.json', recursive=True)[0]
    root = os.path.dirname(xyz)
    os.makedirs('/content/visionProyecto/datasets', exist_ok=True)
    if os.path.abspath(root) != os.path.abspath(DEST):
        if os.path.exists(DEST):
            shutil.rmtree(DEST)
        shutil.move(root, DEST)

print('training_xyz.json:', os.path.exists(os.path.join(DEST, 'training_xyz.json')))
print('imagenes:', len(glob.glob(os.path.join(DEST, 'training', 'rgb', '*.jpg'))))

In [ ]:
# Entrena en GPU (precision mixta automatica). ~pocos minutos en T4.
!python modelo_scratch/entrenar_regresion.py --epocas 25 --lote 64

In [ ]:
# Muestra las curvas y el MAE por dedo
import json
from IPython.display import Image, display

display(Image('resultados/regresion_curvas_entrenamiento.png'))
print(json.dumps(json.load(open('modelos/cnn_regresion_historial.json'))['mae_por_dedo'], indent=2))

In [ ]:
# Descarga el modelo entrenado a tu PC
from google.colab import files
files.download('modelos/cnn_regresion_dedos.keras')